### Try to find URLs for zotero entries that are missing them

Perplexity dialog makrdown has references that are described only by URL only, so I have to  use urls as a key to find the matching
entry in the zotero DB.  But about 150 of 1700 entries are missing URLs (jan 2025), and many are the 
new ones I want for refwrangle.

I tried getting the URLs by having perplexity read a csv file of items w/ no URL, and this was hopeless.
It (gpt01) could do the job, but kept trying to quit early, or to shrug it off.  Super annoying.

So I tried to find titles for the perplexity output URLs and then match by title.  This was
far more successful, but I had a lot of problem with websites not responding to my queries.
Maybe they have screen scraping prevention of some kind...

Then I realized that the output of the **Save my ChatGPT extension** actually has (partial) titles 
in its Sources section.  I think I'll just use that extension 
**instead of messing with raw perplexity output**

In [1]:
fimport pathlib as pl
from pyzotero import zotero
from collections import defaultdict
import pandas as pd
import sys
from urllib.parse import urlparse, urlunparse
from icecream import ic

refwrangle_dir = pl.Path('~/ref/refwrangle').expanduser() # can't reliably get dir of an .ipynb 
sys.path.append(str(refwrangle_dir))
import refwrangle as rfw
import re

%load_ext autoreload
%autoreload 2

In [2]:
outfile = rfw.refwrangle_test_dir / 'tmp' / 'entries_no_url.csv' # goes to perplexity gpto1

### Find entries with missing URLs

In [ ]:

zot = zotero.Zotero(rfw.library_id, rfw.library_type, rfw.api_key)
parentItems = zot.everything(zot.top())
def make_entry_row(parent):
    pdat = parent['data']
    citekeyThis = rfw.get_citation_key(pdat)
    cdict = {column:pdat.get(column) for column in ['date', 'itemType']}
    
    return cdict | dict(citekey=citekeyThis, zotkey=parent['key'], 
                     author = rfw.get_creator(parent), title=rfw.get_title(parent))

no_url_entries, url_entries = [], []
citekeysForURL = defaultdict(list)
for parent in parentItems:
    pdat = parent['data']
    if (itemType := pdat['itemType']) == 'note':
        continue

    item_row = make_entry_row(parent)

    if url := pdat.get('url'): 
        url_entries.append(item_row | {'url':url})
    else:
        no_url_entries.append(item_row)

no_url_entries = pd.DataFrame(no_url_entries)
print(f"\n{len(no_url_entries)=} of {len(parentItems)} entries are missing URls\n")
url_entries = pd.DataFrame(url_entries)
ic(outfile)
no_url_entries.to_csv(outfile)
display(no_url_entries.head(), url_entries.head())

### Try to get article titles for those URLs
This isn't with AI, but doing my own web page requests, followined by old school string-matching

In [38]:
import requests
from bs4 import BeautifulSoup
from requests.exceptions import Timeout, RequestException
import time

def get_webpage_title(url, timeout=5, max_attempts=5, verbose=False):
    """Get the title of the page with a given URL"""
    for attempt in range(max_attempts):
        try:
            response = requests.get(url, timeout=timeout)
            response.raise_for_status()  # Raises an HTTPError for bad responses
            soup = BeautifulSoup(response.content, 'html.parser')
            return soup.title.string.strip() if soup.title else None
        except Timeout:
            if verbose:
                print(f"Request timed out in {timeout=} seconds for URL: {url} (Attempt {attempt + 1}/{max_attempts})")
        except RequestException as e:
            if verbose:
                print(f"An error occurred while fetching URL {url}: {e} (Attempt {attempt + 1}/{max_attempts})")
        
        if attempt < max_attempts - 1:
            time.sleep(2 ** attempt)  # Exponential backoff
    if verbose:
        print(f"Failed to fetch title for URL: {url} after {max_attempts} attempts")
    
    return None

def find_best_zotero_match(webpage_title, zotero_library):
    """Find the zotero entry with a title that best matches a (found) web page title"""
    best_match = None
    best_score = 0

    for item in zotero_library.items():
        item_title = item['data'].get('title', '')
        is_match, score = rfw.match_titles(webpage_title, item_title)
        
        if score > best_score and score > 0.7:  # Adjust threshold as needed
            best_score = score
            best_match = item

    return best_match

### Test web/zotero title matching
On zotero entries where I already have the URL and title, so I know what the right answer is.

*On the 1st and only test, I'm shooting 5 out of 10 successes*

In [39]:
max_tests=10
results = []
for _, search_item in url_entries.copy().iterrows(): # test items supplying the URLs you're looking for 
    if (ntests := len(results)) >= max_tests:
        print(f'Done testing {max_tests=}')
        break
    ic(ntests, search_item.zotkey, search_item.title)
    found_web_title = get_webpage_title(search_item.url)
    search_item['found_web_title'] = found_web_title
    if found_web_title:
        ic(found_web_title)
        match_item = find_best_zotero_match(found_web_title, zot)
        if match_item:
            search_item['key_best_zot'] = match_item['key']
            if pdat :=  match_item['data']:
                search_item['title_best_zot']=pdat.get('title')
                ic(search_item['title_best_zot'])
    results.append(search_item)

results = pd.DataFrame(results)

ic| ntests: 0
    search_item.zotkey: '6V73VWME'
    search_item.title: ("Descriptive social norms and motivation to vote: Everybody's voting and so "
                        'should you')
ic| ntests: 1
    search_item.zotkey: '6VPD3STC'
    search_item.title: ('Political advertising in the 2012 presidential election: How visual and '
                        'aural techniques are used to convey meaning')
ic| found_web_title: ('"Political Advertising in the 2012 Presidential Election: How Visual an" by '
                      'James Lake Poston')
ic| search_item['title_best_zot']: ('Political advertising in the 2012 presidential election: How visual and '
                                    'aural techniques are used to convey meaning')
ic| ntests: 2
    search_item.zotkey: 'YIN5FL7M'
    search_item.title: 'An exploration of global altruistic variations by country'
ic| found_web_title: 'OhioLINK ETD: Deaton, Eddie W.'
ic| search_item['title_best_zot']: 'How minds change: The surprising

Done testing max_tests=10


In [40]:
results = pd.DataFrame(results)
results

,date,itemType,citekey,zotkey,author,title,url,found_web_title,key_best_zot,title_best_zot
0,2009,journalArticle,Gerber09normsMotiveVote,6V73VWME,"Gerber, Alan S",Descriptive social norms and motivation to vot...,https://www.journals.uchicago.edu/doi/abs/10.1...,None,NaN,NaN
1,2013-08,thesis,Poston13politAdsVizAuralMeaning,6VPD3STC,"Poston, James Lake",Political advertising in the 2012 presidential...,https://scholarworks.boisestate.edu/td/602/,"""Political Advertising in the 2012 Presidentia...",6VPD3STC,Political advertising in the 2012 presidential...
2,2021-11,thesis,Deaton21altruisByCountry,YIN5FL7M,"Deaton, Eddie William",An exploration of global altruistic variations...,https://rave.ohiolink.edu/etdc/view?acc_num=xu...,"OhioLINK ETD: Deaton, Eddie W.",2F4J9AL2,How minds change: The surprising science of be...
3,2017,conferencePaper,Calafiore17newsTopicSparseLrnPresid,KBL5IGVD,"Calafiore, Giuseppe C.",Topic analysis in news via sparse learning: a ...,https://www.sciencedirect.com/science/article/...,None,NaN,NaN
4,2023,book,Jarvis23gutenbergInternet,ETBJL3D9,"Jarvis, Jeff",The Gutenberg Parenthesis: The age of print an...,https://www.bloomsbury.com/us/gutenberg-parent...,None,NaN,NaN
5,2023,book,Young23wrongAppetiteMisinfo,EVVT7PL7,"Young, Dannagal Goldthwaite","Wrong: How media, politics, and identity drive...",https://www.press.jhu.edu/books/title/12834/wrong,Wrong | Hopkins Press,EVVT7PL7,"Wrong: How media, politics, and identity drive..."
6,2022-04-05,book,Ripley22conflictTrapOut,UTU4PRME,"Ripley, Amanda",High conflict: Why we get trapped and how we g...,https://www.simonandschuster.com/books/High-Co...,High Conflict | Book by Amanda Ripley | Offici...,UTU4PRME,High conflict: Why we get trapped and how we g...
7,2022-03-08,book,Hasen22cheapSpchDisinfoCure,QMA5DB7E,"Hasen, Richard L.",Cheap speech: How disinformation poisons our p...,https://yalebooks.yale.edu/9780300274097/cheap...,Cheap Speech,QMA5DB7E,Cheap speech: How disinformation poisons our p...
8,APRIL 2023,report,IAS23takingActionAttentionVI,9ZYY2IMD,"IAS,",Taking Action on Attention I,https://integralads.com/insider/taking-action-...,None,NaN,NaN
9,July 2024,report,IAS24takingActionAttentionVII,XLSSHSL8,"IAS,",Taking Action on Attention II,https://integralads.com/insider/taking-action-...,None,NaN,NaN


In [41]:
for ix, res in results.iterrows():
    if res.found_web_title:
        print(ix)
        print(res.title)
        print(res.found_web_title)


1
Political advertising in the 2012 presidential election: How visual and aural techniques are used to convey meaning
"Political Advertising in the 2012 Presidential Election: How Visual an" by James Lake Poston
2
An exploration of global altruistic variations by country
OhioLINK ETD: Deaton, Eddie W.
5
Wrong: How media, politics, and identity drive our appetite for misinformation
Wrong | Hopkins Press
6
High conflict: Why we get trapped and how we get out
High Conflict | Book by Amanda Ripley | Official Publisher Page | Simon & Schuster
7
Cheap speech: How disinformation poisons our politics―and how to cure it
Cheap Speech
